In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
# import seaborn as sns
# import matplotlib.pyplot as plt

import ast
import re
import shutil


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [4]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(99, 11)


,id,template,placeholders,session,cam,constructed_filename,label,top1_pred,correct_top1,top5_preds,correct_top5
74,261008-114415,Bending [something] so that it deforms,cardboard,bending_1,cam2,261008-114415_cam2.avi,2,2,True,"[129, 16, 3, 143, 2]",True
27,261008-112813,Pretending to sprinkle air onto [something],nothing,mime_pretend_sprinkle,cam0,261008-112813_cam0.avi,79,68,False,"[67, 160, 45, 79, 68]",True


In [5]:
# rename top1_label to top1_pred
result = result.rename(columns={'id':'video_id'})

del result['constructed_filename']

result.sample(3)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
56,261008-113428,Pretending to pick [something] up,wallet,pretend_pickUp_1,cam2,68,100,False,"[56, 55, 68, 67, 100]",True
4,261008-111608,Pretending to poke [something],nothing,mime_pretend_poke,cam1,69,69,True,"[160, 68, 56, 45, 69]",True
44,261008-113021,Pretending to sprinkle air onto [something],wallet,pretend_sprinkle_1,cam2,79,56,False,"[100, 15, 55, 67, 56]",False


In [6]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(99, 174)


NpzFile './predictions.npz' with keys: logits, probs, video_ids, labels

# main result

In [7]:

n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()

print(f"Number of samples: {n}")
print(f"Top-1 accuracy: {top1_acc:.4f}")
print(f"Top-5 accuracy: {top5_acc:.4f}")

Number of samples: 99
Top-1 accuracy: 0.3131
Top-5 accuracy: 0.5152


- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [8]:
result.groupby("cam")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
cam,
cam0,0.24
cam1,0.52
cam2,0.18


In [10]:
result.groupby(['cam']).agg(
    # count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,top1_acc,top5_acc
cam,,
cam0,0.24,0.45
cam1,0.52,0.76
cam2,0.18,0.33


In [11]:
result.groupby(['template']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
template,,,
Bending [something] so that it deforms,18,0.39,0.56
Moving [something] and [something] closer to each other,18,0.33,0.39
Pretending to pick [something] up,18,0.39,0.67
Pretending to poke [something],27,0.11,0.41
Pretending to sprinkle air onto [something],18,0.44,0.61


In [12]:
result[result.cam=='cam1'].groupby(['template','session']).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

count  top1_acc  top5_acc
template                                                session                                              
Bending [something] so that it deforms                  bending_1                       3      0.67      1.00
                                                        mime_bending                    3      0.00      0.33
Moving [something] and [something] closer to each other mime_moveCloserToEachOther      3      0.00      0.00
                                                        moveCloserToEachOther_1         3      1.00      1.00
Pretending to pick [something] up                       mime_pretend_pickUp             3      1.00      1.00
                                                        pretend_pickUp_1                3      1.00      1.00
Pretending to poke [something]                          mime_pretend_poke               6      0.33      0.50
                                                        pretend_poke_1                  3      0.00      1.00
Pretending to sprinkle air onto [something]             mime_pretend_sprinkle           3      1.00      1.00
                                                        pretend_sprinkle_1              3      0.33      1.00

In [18]:
temp = result[result.cam=='cam1'].copy()

temp.groupby( np.where(temp['session'].str.startswith('mime'), 'mime', 'control') ).agg(
    count=('video_id', 'count'),
    top1_acc=('correct_top1', 'mean'),
    top5_acc=('correct_top5', 'mean')
)

,count,top1_acc,top5_acc
control,15,0.60,1.00
mime,18,0.44,0.56


# Granular analysis

##### id2label

In [19]:
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [20]:
result.head(1)

,video_id,template,placeholders,session,cam,label,top1_pred,correct_top1,top5_preds,correct_top5
0,261008-111314,Pretending to poke [something],nothing,mime_pretend_poke,cam0,69,100,False,"[35, 22, 160, 69, 100]",True


In [21]:
df = result[['video_id',	'cam'	,'session'	,'placeholders','template',	'label'	,'top1_pred','correct_top1','correct_top5']]
print(df.shape)
df.head(2)

(99, 9)


,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5
0,261008-111314,cam0,mime_pretend_poke,nothing,Pretending to poke [something],69,100,False,True
1,261008-111314,cam1,mime_pretend_poke,nothing,Pretending to poke [something],69,100,False,True


In [22]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,placeholders,template,label,top1_pred,correct_top1,correct_top5,prob_0,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,261008-111314,cam0,mime_pretend_poke,nothing,Pretending to poke [something],69,100,False,True,0.000422,...,0.000001,0.000030,0.000030,0.000008,0.000018,0.000000,0.000162,0.000012,0.000003,0.000002
1,261008-111314,cam1,mime_pretend_poke,nothing,Pretending to poke [something],69,100,False,True,0.000017,...,0.000004,0.000020,0.000005,0.000001,0.000008,0.000000,0.000105,0.000010,0.000002,0.000002


#### Main Analysis 


In [23]:
pd.set_option('display.float_format',  '{:,.4f}'.format)

In [24]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        # top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [25]:
df.session.unique()

array(['mime_pretend_poke', 'pretend_poke_1', 'mime_pretend_sprinkle',
       'pretend_sprinkle_1', 'mime_pretend_pickUp', 'pretend_pickUp_1',
       'mime_bending', 'bending_1', 'mime_moveCloserToEachOther',
       'moveCloserToEachOther_1'], dtype=object)

In [26]:
avg_prob_in_diff_cases(df, session_type='bending', label='Bending [something] so that it deforms', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= bending =========================
label 2: Bending [something] so that it deforms
cams used: ['cam1']
        session  videos  prob_mean  prob_std  top1acc                                     examples
0     bending_1       3     0.6086    0.4311   0.6667  261008-114415, 261008-114428, 261008-114453
1  mime_bending       3     0.0490    0.0492   0.0000  261008-114331, 261008-114344, 261008-114350
==================== Videos ====================
         video_id   cam       session placeholders                                template label top1_pred correct_top1 correct_top5 prob_65 prob_79 prob_150 prob_3 prob_91
64  261008-114331  cam1  mime_bending      nothing  Bending [something] so that it deforms     2        65        False        False  0.4020  0.1231   0.1081 0.0788  0.0369

Top 5 Probability Breakdown:
  prob_65  ->  0.40195  |  Pretending to be tearing [something that is not tearable]
  prob_79  ->  0.12306  |  Pretending to sprinkle air onto [somet

In [27]:
avg_prob_in_diff_cases(df, session_type='Closer', label='Moving [something] and [something] closer to each other', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= Closer =========================
label 37: Moving [something] and [something] closer to each other
cams used: ['cam1']
                      session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_moveCloserToEachOther       3     0.0026    0.0021   0.0000  261008-114837, 261008-114847, 261008-114901
1     moveCloserToEachOther_1       3     0.9578    0.0356   1.0000  261008-114921, 261008-114942, 261008-114959
==================== Videos ====================
         video_id   cam                     session      placeholders                                                 template label top1_pred correct_top1 correct_top5 prob_93 prob_109 prob_129 prob_74 prob_100
82  261008-114837  cam1  mime_moveCloserToEachOther  nothing, nothing  Moving [something] and [something] closer to each other    37        93        False        False  0.2136   0.1046   0.0952  0.0757   0.0663

Top 5 Probability Breakdown:
  prob_93  ->

In [28]:
avg_prob_in_diff_cases(df, session_type='pick', label='Pretending to pick [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= pick =========================
label 68: Pretending to pick [something] up
cams used: ['cam1']
               session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_pretend_pickUp       3     0.7021    0.2021   1.0000  261008-113356, 261008-113406, 261008-113415
1     pretend_pickUp_1       3     0.8991    0.0950   1.0000  261008-113428, 261008-113443, 261008-113455
==================== Videos ====================
         video_id   cam              session placeholders                           template label top1_pred correct_top1 correct_top5 prob_68 prob_79 prob_81 prob_129 prob_80
46  261008-113356  cam1  mime_pretend_pickUp      nothing  Pretending to pick [something] up    68        68         True         True  0.9037  0.0535  0.0171   0.0072  0.0035

Top 5 Probability Breakdown:
  prob_68  ->  0.90368  |  Pretending to pick [something] up
  prob_79  ->  0.05354  |  Pretending to sprinkle air onto [something

In [29]:
avg_prob_in_diff_cases(df, session_type='poke', label='Pretending to poke [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= poke =========================
label 69: Pretending to poke [something]
cams used: ['cam1']
             session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_pretend_poke       6     0.0915    0.1098   0.3333  261008-111314, 261008-111608, 261008-111617
1     pretend_poke_1       3     0.0557    0.0424   0.0000  261008-112131, 261008-112156, 261008-112220
==================== Videos ====================
        video_id   cam            session placeholders                        template label top1_pred correct_top1 correct_top5 prob_100 prob_68 prob_160 prob_69 prob_79
1  261008-111314  cam1  mime_pretend_poke      nothing  Pretending to poke [something]    69       100        False         True   0.3783  0.1071   0.0998  0.0884  0.0398

Top 5 Probability Breakdown:
  prob_100 ->  0.37831  |  Pushing [something] so that it slightly moves
  prob_68  ->  0.10706  |  Pretending to pick [something] up
  prob_160 ->  

In [30]:
avg_prob_in_diff_cases(df, session_type='sprinkle', label='Pretending to sprinkle air onto [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= sprinkle =========================
label 79: Pretending to sprinkle air onto [something]
cams used: ['cam1']
                 session  videos  prob_mean  prob_std  top1acc                                     examples
0  mime_pretend_sprinkle       3     0.8796    0.1192   1.0000  261008-112813, 261008-112823, 261008-112834
1     pretend_sprinkle_1       3     0.2402    0.3109   0.3333  261008-112944, 261008-113003, 261008-113021
==================== Videos ====================
         video_id   cam                session placeholders                                     template label top1_pred correct_top1 correct_top5 prob_79 prob_68 prob_27 prob_45 prob_129
28  261008-112813  cam1  mime_pretend_sprinkle      nothing  Pretending to sprinkle air onto [something]    79        79         True         True  0.7503  0.1768  0.0228  0.0113   0.0103

Top 5 Probability Breakdown:
  prob_79  ->  0.75029  |  Pretending to sprinkle air onto [something]
  prob_68  ->  